# 00 — Setup de la sesión de Colab

Corré esto primero en cada sesión. Colab arranca de cero cada vez: monta Drive,
clona el repo, instala las herramientas y verifica que arranquen de verdad.

Es el equivalente de `check_env.sh`, recortado a lo que Colab necesita —acá solo
se descarga y se valida, no se alinea— así que no hacen falta bowtie, fastp ni
ViennaRNA.

## Preámbulo: montar Drive y clonar el repo

El repo es público, así que el clon no necesita credenciales. **Los notebooks
llaman a los scripts del repo en vez de reimplementarlos**: el criterio de
selección de corridas y el de verificación de ensamblados tienen que vivir en
un solo lugar, o dejan de ser reproducibles.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib
DRIVE = pathlib.Path('/content/drive/MyDrive/tesis')
CLON  = pathlib.Path('/content/tesis')
assert DRIVE.exists(), f'no veo {DRIVE} — ¿montaste la cuenta correcta?'
print('Drive OK:', DRIVE)

In [ ]:
import subprocess

REPO = 'youkonskernel-afk/tesis'
URL_ANON = 'https://github.com/' + REPO + '.git'

_AYUDA = (
    "No pude clonar de forma anonima y no hay GITHUB_TOKEN en los Secrets.",
    "Dos salidas, cualquiera sirve:",
    "  a) hacer el repo publico: Settings -> General -> Change visibility",
    "  b) crear un PAT de solo lectura y guardarlo como GITHUB_TOKEN en el",
    "     panel de Secrets de Colab (la llave a la izquierda), habilitando",
    "     el acceso para este notebook.",
)


def _sin_token(txt, secreto):
    # git incluye la URL en sus mensajes de error, y esa URL lleva el token.
    return txt.replace(secreto, '***') if secreto else txt


def clonar():
    if CLON.exists():
        r = subprocess.run(['git', '-C', str(CLON), 'pull', '--ff-only'],
                           capture_output=True, text=True)
        return 'ya estaba clonado; ' + (r.stdout.strip() or r.stderr.strip())

    # 1. Anonimo. Alcanza si el repo es publico.
    r = subprocess.run(['git', 'clone', '--depth', '1', URL_ANON, str(CLON)],
                       capture_output=True, text=True)
    if r.returncode == 0:
        return 'clon anonimo (el repo es publico)'

    # 2. Con token de los Secrets de Colab. Para repo privado.
    tok = None
    try:
        from google.colab import userdata
        tok = userdata.get('GITHUB_TOKEN')
    except Exception:
        pass
    if not tok:
        raise RuntimeError(chr(10).join(_AYUDA))

    url = 'https://x-access-token:' + tok + '@github.com/' + REPO + '.git'
    r = subprocess.run(['git', 'clone', '--depth', '1', url, str(CLON)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('el clon con token fallo: ' + _sin_token(r.stderr, tok))

    # Sin esto el token queda escrito en .git/config dentro de la VM.
    subprocess.run(['git', '-C', str(CLON), 'remote', 'set-url', 'origin', URL_ANON],
                   capture_output=True, text=True)
    return 'clon con token (el repo es privado)'


print(clonar())
print(subprocess.run(['git', '-C', str(CLON), 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout.strip())

## Herramientas

`sra-tools` trae `prefetch` y `vdb-validate`. Se baja el tarball oficial de NCBI
en vez de usar `apt`, porque el paquete de Ubuntu suele ir varias versiones
atrás. Si la URL cambia, ajustá `SRA_VER`.

El entorno `srna2` del proyecto pinea sra-tools 3.4.1. Acá la versión puede
diferir y no es grave: Colab solo descarga y valida, no produce resultados que
entren en la tesis. Lo que sí importa es que `vdb-validate` exista.

In [ ]:
import glob, os, subprocess, shutil

# La version del tarball puede cambiar o desaparecer. Si falla, se cae a apt:
# es mas vieja, pero aca solo se descarga y se valida, no se produce nada que
# entre en la tesis. Lo unico que importa es que prefetch y vdb-validate corran.
SRA_VER = '3.1.1'
URL = f'https://ftp-trace.ncbi.nlm.nih.gov/sra/sdk/{SRA_VER}/sratoolkit.{SRA_VER}-ubuntu64.tar.gz'


def sh(cmd, t=600):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=t)


def por_tarball():
    r = sh(f'wget -q -O /tmp/sra.tar.gz "{URL}"')
    if r.returncode != 0:
        return None, f'no pude bajar el tarball ({URL})'
    r = sh('tar -xzf /tmp/sra.tar.gz -C /opt')
    if r.returncode != 0:
        return None, 'el tarball bajo pero no se desempaqueto'
    c = glob.glob(f'/opt/sratoolkit.{SRA_VER}*/bin')
    return (c[0], None) if c else (None, 'no encontre el bin/ tras desempaquetar')


def por_apt():
    r = sh('apt-get -qq install -y sra-toolkit')
    if r.returncode != 0:
        return None, 'apt-get tambien fallo'
    ruta = shutil.which('prefetch')
    return (os.path.dirname(ruta), None) if ruta else (None, 'apt no dejo prefetch en PATH')


sh('apt-get -qq update')
sh('apt-get -qq install -y jq')

ruta, err = por_tarball()
if ruta:
    print(f'sra-tools {SRA_VER} desde el tarball oficial: {ruta}')
else:
    print(f'tarball no sirvio ({err}); probando apt...')
    ruta, err2 = por_apt()
    if ruta:
        print(f'sra-tools desde apt (version distinta de la del tarball): {ruta}')
    else:
        raise RuntimeError(
            'No pude instalar sra-tools por ninguna via.\n'
            f'  tarball: {err}\n'
            f'  apt    : {err2}\n'
            'Revisa si cambio la version en https://github.com/ncbi/sra-tools/wiki '
            'y ajusta SRA_VER, o instalalo a mano en esta celda.')

if ruta not in os.environ['PATH']:
    os.environ['PATH'] = ruta + ':' + os.environ['PATH']

## Verificar que arrancan

No alcanza con que el binario exista: tiene que ejecutar. Es la misma lógica que
`check_env.sh` aplica en la máquina local.

In [ ]:
import shutil, subprocess

def prueba(cmd, args=['--version']):
    ruta = shutil.which(cmd)
    if not ruta:
        return f'[MAL] {cmd}: no está en PATH'
    try:
        r = subprocess.run([cmd] + args, capture_output=True, text=True, timeout=60)
        v = (r.stdout + r.stderr).strip().split('\n')[0]
        return f'[OK ] {cmd}: {v}'
    except Exception as e:
        return f'[MAL] {cmd}: no arranca ({e})'

for c in ['prefetch', 'vdb-validate', 'curl', 'jq', 'git']:
    print(prueba(c))

## Árbol de Drive

In [ ]:
esperadas = ['00_manifiestos','10_bam','20_yasma','30_qc','40_features',
             '50_modelos','60_figuras','70_genomas','80_sra']
for d in esperadas:
    p = DRIVE / d
    print(f"[{'OK ' if p.exists() else 'FALTA'}] {d}")

import shutil as _sh
libre = _sh.disk_usage('/content').free / 1e9
print(f'\ndisco efímero de la VM: {libre:.0f} GB libres')

## Después de esto

- `descarga_genomas.ipynb` — verificar y bajar los ensamblados
- `10_descarga_runs.ipynb` — resolver el manifiesto y bajar los `.sra`
- `90_estado.ipynb` — ver qué falta